# 03) Modélisation du prix des EHPAD

**Objectif :** prédire `PRIX_MOIS_HEBPERM_CS_GIR56` (prix mensuel de l'hébergement permanent, chambre seule) à partir des variables retenues dans le notebook 02.

Plan :
>
0. Chargement
1. Découpage train/test par gestionnaire
2. Premier modèle : le loyer seul
3. Régressions linéaires par blocs de variables
4. Random Forest
5. Gradient Boosting
6. Gradient Boosting avec la cible en log
7. Validation croisée par gestionnaire
8. Analyse des EHPAD les plus mal prédits
9. Conclusion
10. Limites et pistes

## 0. Chargement

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Graine fixée : mêmes découpages et mêmes résultats à chaque exécution
SEED = 42

# Chargement du df
df = pd.read_csv('../data/clean/modele.csv', dtype={'finesset': str, 'finessej': str, 'DEP': str})

# Population de l'EPCI en log (écart très important entre petits EPCI et métropoles)
df['P19_POP_epci'] = np.log(df['P19_POP_epci'])

cible = 'PRIX_MOIS_HEBPERM_CS_GIR56'
df

,finesset,finessej,PRIX_MOIS_HEBPERM_CS_GIR56,STATUT_JUR,ASH,nb_capins_HP,en_groupe,anciennete,loypredm2,AGE6Q219_epci,AGE6PPAT19_epci,P19_POP_epci,P19_POP75P_perc_epci,taux_equip_epci,DEP
0,010002228,750057622,3154.8,3-Privé commercial,NON,60,False,13.0,10.074507,22130.0,16.2,11.278000,7.689050,86.853623,01
1,010004059,010783009,2204.4,2-Privé non lucratif,OUI,66,True,11.0,20.693334,31890.0,18.3,11.495342,4.814770,80.535243,01
2,010006799,690802715,2092.2,2-Privé non lucratif,OUI,80,True,8.0,10.984294,22010.0,18.5,11.195677,9.263305,74.721181,01
3,010008571,010007987,1561.5,1-Public,OUI,69,False,10.0,7.996378,21500.0,16.9,11.056667,8.311903,105.756070,01
4,010009223,010787109,2189.7,2-Privé non lucratif,OUI,75,True,3.0,8.931485,21500.0,16.9,11.056667,8.311903,105.756070,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6668,950807826,920031267,2843.4,3-Privé commercial,NON,90,False,28.0,16.328407,26550.0,19.1,12.117924,7.643977,89.726898,95
6669,950808469,950014738,2178.3,3-Privé commercial,NON,60,True,25.0,13.652158,26730.0,21.6,10.117267,6.220433,38.945498,95
6670,950808519,950808501,2974.8,3-Privé commercial,NON,85,False,27.0,15.619351,23780.0,11.7,12.266217,3.801350,62.794229,95
6671,950808956,250015658,3137.4,3-Privé commercial,NON,103,True,23.0,16.101724,25920.0,14.3,12.549762,6.875349,74.418238,95


## 1. Découpage train/test par gestionnaire

Les EHPAD d'un même gestionnaire ont des prix très proches (écart-type médian de 68 € contre 522 € sur toute la base). Avec un découpage aléatoire, le modèle verrait des établissements quasi identiques à ceux du test, et ses scores seraient trop optimistes. On découpe donc **par gestionnaire** : tous les EHPAD d'un même gestionnaire sont soit dans le train, soit dans le test.

Le découpage est fait **une seule fois**, avec une graine fixée : tous les modèles du notebook sont évalués sur le même jeu de test.

In [2]:
# Transformer les variables qualitatives en colonnes de 0 et 1 (one-hot)
modele_encode = pd.get_dummies(df,
                               columns=['STATUT_JUR', 'ASH', 'DEP'],  # variables qualitatives à encoder
                               drop_first=True,                        # supprime une catégorie de référence par variable
                               dtype=int)                              # des 0 et 1 plutôt que True/False

# Découpage 80/20 : les EHPAD d'un même gestionnaire sont soit dans le train, soit dans le test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(df, groups=df['finessej']))

# Version lisible (variables qualitatives en clair) et version encodée, avec les mêmes lignes
modele_train, modele_test = df.iloc[train_idx], df.iloc[test_idx]
train, test = modele_encode.iloc[train_idx], modele_encode.iloc[test_idx]

# Cible du train et du test
y_train, y_test = train[cible], test[cible]

print("shape du jeu d'entraînement : ", train.shape)
print("shape du jeu de test : ", test.shape)
print(f"Part du test : {round(len(test) / len(df) * 100, 1)} %")

shape du jeu d'entraînement :  (5446, 110)
shape du jeu de test :  (1227, 110)
Part du test : 18.4 %


> 18,4 % des EHPAD sont dans le test. La répartition n'est pas exactement 80/20, car le découpage se fait par gestionnaire et non par établissement.

## 2. Premier modèle : uniquement avec la variable loyer (`loypredm2`)

In [3]:
modele_1 = LinearRegression().fit(modele_train[['loypredm2']], y_train)
print(f"Le prix prédit vaut environ {round(modele_1.intercept_)} € + {round(modele_1.coef_[0])} € par euro de loyer au m²")

Le prix prédit vaut environ 1190 € + 93 € par euro de loyer au m²


In [4]:
# Baseline : prédiction du prix médian de chaque case statut x ASH, appris sur le train
medianes = modele_train.groupby(['STATUT_JUR', 'ASH'])[cible].median()

# Pour chaque EHPAD du test, on va chercher la médiane de sa case
pred_baseline = modele_test.set_index(['STATUT_JUR', 'ASH']).index.map(medianes)
print('MAE baseline (statut x ASH) :', round(mean_absolute_error(y_test, pred_baseline)), '€')

# Référence encore plus simple : la médiane globale du train pour tout le monde
pred_naive = [y_train.median()] * len(y_test)
print('MAE médiane globale :', round(mean_absolute_error(y_test, pred_naive)), '€')

MAE baseline (statut x ASH) : 216 €
MAE médiane globale : 334 €


In [5]:
test_pred_modele_1 = modele_1.predict(modele_test[['loypredm2']])

print('R²  ->', round(r2_score(y_test, test_pred_modele_1), 2))
print('MSE ->', round(mean_squared_error(y_test, test_pred_modele_1)))
print('MAE ->', round(mean_absolute_error(y_test, test_pred_modele_1)), '€')

R²  -> 0.4
MSE -> 139057
MAE -> 285 €


> - Avec le loyer seul, le modèle explique 40 % des écarts de prix (R² de 0,40).
> - Son erreur moyenne (285 €) est moins bonne que la baseline statut × ASH (216 €), mais meilleure que la médiane globale (334 €).
> - Le statut et l'ASH sont indispensables pour avoir un modèle plus robuste.

## 3. Régressions linéaires par blocs de variables

On ajoute les variables par blocs pour mesurer ce que chacun apporte. Les variables qualitatives (`STATUT_JUR`, `ASH`, `DEP`) ont été encodées en colonnes de 0 et 1 (one-hot) à l'étape 1.

In [6]:
# Listes de colonnes de chaque bloc
statut_ash = [c for c in modele_encode.columns if c.startswith('STATUT_JUR_') or c.startswith('ASH_')]
autres = ['nb_capins_HP', 'en_groupe', 'anciennete', 'AGE6Q219_epci', 'AGE6PPAT19_epci',
          'P19_POP_epci', 'P19_POP75P_perc_epci', 'taux_equip_epci']
dep = [c for c in modele_encode.columns if c.startswith('DEP_')]

# Les 4 modèles : chaque bloc s'ajoute au précédent
blocs = {
    '1. Loyer seul': ['loypredm2'],
    '2. + statut et ASH': ['loypredm2'] + statut_ash,
    '3. + autres variables': ['loypredm2'] + statut_ash + autres,
    '4. + département': ['loypredm2'] + statut_ash + autres + dep,
}

# Entraîner chaque modèle et mesurer ses erreurs sur le test
resultats = []
for nom, colonnes in blocs.items():
    modele_lr = LinearRegression().fit(train[colonnes], y_train)   # entraînement sur le train
    prediction = modele_lr.predict(test[colonnes])                 # prédiction sur le test
    resultats.append({
        'Modèle': nom,
        'Nb variables': len(colonnes),
        'MAE (€)': round(mean_absolute_error(y_test, prediction)),
        'R²': round(r2_score(y_test, prediction), 2),
    })

pd.DataFrame(resultats)

,Modèle,Nb variables,MAE (€),R²
0,1. Loyer seul,1,285,0.40
1,2. + statut et ASH,4,190,0.70
2,3. + autres variables,12,189,0.70
3,4. + département,107,175,0.73


> **Résultats :**
> - **Le statut et l'ASH font l'essentiel du travail :** l'erreur passe de 285 € à 190 €, sous la baseline de 216 €.
> - **Les 8 autres variables apportent peu** en régression linéaire (1 € gagnés) : leur information recoupe celle du loyer et du statut.
> - **Le département apporte un vrai gain** (14 €), ce qui confirme l'exploration et le rôle du dpt dans les stratégies liés aux ehpad
>
> Meilleur modèle linéaire : **175 € d'erreur moyenne, R² de 0,73**.

## 4. Random Forest

Une forêt aléatoire combine 300 arbres de décision. Contrairement à la régression linéaire, elle peut capter des effets non linéaires et des interactions (par exemple un loyer qui ne joue pas de la même façon selon le statut).

In [ ]:
# Toutes les variables du modèle : tout sauf les identifiants et la cible
variables = [col for col in modele_encode.columns if col not in ['finesset', 'finessej', cible]]

# Création de la forêt aléatoire
rf = RandomForestRegressor(n_estimators=300,      # 300 arbres de décision
                           min_samples_leaf=5,    # au moins 5 EHPAD par feuille, pour limiter le surapprentissage
                           random_state=SEED,     
                           n_jobs=-1)             

rf.fit(train[variables], y_train)

# Erreurs sur le test
prediction_rf = rf.predict(test[variables])
print('MAE test :', round(mean_absolute_error(y_test, prediction_rf)), '€')
print('R² test  :', round(r2_score(y_test, prediction_rf), 2))

# Erreurs sur le train, pour repérer un éventuel surapprentissage
prediction_train = rf.predict(train[variables])
print('MAE train :', round(mean_absolute_error(y_train, prediction_train)), '€')
print('R² train  :', round(r2_score(y_train, prediction_train), 2))

# Les 10 variables les plus utiles à la forêt
pd.Series(rf.feature_importances_, index=variables).sort_values(ascending=False).head(10).round(3)

MAE test : 162 €
R² test  : 0.76
MAE train : 105 €
R² train  : 0.91


STATUT_JUR_3-Privé commercial      0.550
loypredm2                          0.259
AGE6Q219_epci                      0.057
anciennete                         0.021
nb_capins_HP                       0.020
STATUT_JUR_2-Privé non lucratif    0.015
P19_POP75P_perc_epci               0.013
taux_equip_epci                    0.011
P19_POP_epci                       0.011
AGE6PPAT19_epci                    0.011
dtype: float64

>la forêt fait mieux que la régression linéaire (162 € contre 175 €, R² de 0,76). L'écart entre train (105 €) et test (162 €) montre un surapprentissage modéré.
>
> Les variables les plus utilisées sont le statut privé commercial et le loyer, ce qui confirme l'exploration. Le département pèse peu ici : en one-hot, il est éclaté en 95 colonnes presque toujours à 0, que les arbres exploitent mal.

## 5. Gradient Boosting

In [8]:
gb = HistGradientBoostingRegressor(learning_rate=0.05,     # chaque nouvel arbre corrige un peu les erreurs des précédents
                                   max_iter=1000,          # nombre maximum d'arbres
                                   min_samples_leaf=20,    # au moins 20 EHPAD par feuille
                                   early_stopping=True,    # s'arrête quand le modèle ne progresse plus
                                   random_state=SEED)      # même résultat à chaque exécution

gb.fit(train[variables], y_train)

# Erreurs sur le test
prediction_gb = gb.predict(test[variables])
print('MAE test :', round(mean_absolute_error(y_test, prediction_gb)), '€')
print('R² test  :', round(r2_score(y_test, prediction_gb), 2))

# Erreurs sur le train, pour repérer un éventuel surapprentissage
prediction_train_gb = gb.predict(train[variables])
print('MAE train :', round(mean_absolute_error(y_train, prediction_train_gb)), '€')
print('R² train  :', round(r2_score(y_train, prediction_train_gb), 2))

# Nombre d'arbres réellement construits avant l'arrêt
print("Nombre d'arbres :", gb.n_iter_)

MAE test : 160 €
R² test  : 0.76
MAE train : 126 €
R² train  : 0.88
Nombre d'arbres : 170


résultat très proche de la forêt sur le test (160 € contre 162 €, R² de 0,76). Le surapprentissage est plus faible que celui de la forêt : 126 € d'erreur sur le train contre 160 € sur le test.

## 6. Gradient Boosting avec la cible en log

L'exploration a montré que le prix est très asymétrique (asymétrie de 2,19, ramenée à 1,18 avec le log). On entraîne le modèle sur le log du prix, pour que les EHPAD très chers pèsent moins, puis on remet les prédictions en euros avec l'exponentielle.

In [9]:
# Gradient Boosting, mêmes réglages que précédemment
gb_log = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=1000, min_samples_leaf=20,
                                       early_stopping=True, random_state=SEED)

# Entraînement sur le log du prix au lieu du prix en euros
gb_log.fit(train[variables], np.log(y_train))

# Prédictions en log, remises en euros avec exp() (l'inverse du log)
prediction_log = np.exp(gb_log.predict(test[variables]))
prediction_train_log = np.exp(gb_log.predict(train[variables]))

# Erreurs en euros, comparables aux modèles précédents
print('MAE test :', round(mean_absolute_error(y_test, prediction_log)), '€')
print('R² test  :', round(r2_score(y_test, prediction_log), 2))
print('MAE train :', round(mean_absolute_error(y_train, prediction_train_log)), '€')

MAE test : 157 €
R² test  : 0.77
MAE train : 124 €


Sur ce découpage, la cible en log donne 157 € (R² de 0,77), contre 160 € sans le log. Les écarts entre les modèles à arbres sont de quelques euros : un seul découpage ne suffit pas pour les départager. C'est l'objet de la validation croisée ci-dessous.

## 7. Validation croisée par gestionnaire (Cette partie a été entièrement codé par IA pour vérifier davantage mes résultats)

Un seul découpage train/test peut être plus ou moins favorable par hasard. On répète donc l'évaluation sur **5 découpages** (`GroupKFold`) : chaque EHPAD passe une fois dans le test, et les EHPAD d'un même gestionnaire restent toujours ensemble. On regarde la moyenne et l'écart-type de l'erreur sur les 5 plis.

In [ ]:
# Les modèles à comparer : (fonction qui crée le modèle, colonnes utilisées, cible en log ou non)
modeles_cv = {
    'Régression linéaire, loyer seul': (lambda: LinearRegression(), ['loypredm2'], False),
    'Régression linéaire, toutes variables': (lambda: LinearRegression(), variables, False),
    'Random Forest': (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                    random_state=SEED, n_jobs=-1), variables, False),
    'Gradient Boosting': (lambda: HistGradientBoostingRegressor(learning_rate=0.05, max_iter=1000,
                              min_samples_leaf=20, early_stopping=True, random_state=SEED), variables, False),
    'Gradient Boosting, cible en log': (lambda: HistGradientBoostingRegressor(learning_rate=0.05, max_iter=1000,
                              min_samples_leaf=20, early_stopping=True, random_state=SEED), variables, True),
}

# 5 plis, groupés par gestionnaire
gkf = GroupKFold(n_splits=5)
X, y, groupes = modele_encode, modele_encode[cible], modele_encode['finessej']

scores = []
for pli, (idx_tr, idx_te) in enumerate(gkf.split(X, y, groups=groupes)):
    X_tr, X_te, y_tr, y_te = X.iloc[idx_tr], X.iloc[idx_te], y.iloc[idx_tr], y.iloc[idx_te]

    # Baseline statut x ASH, apprise sur le train du pli
    med = df.iloc[idx_tr].groupby(['STATUT_JUR', 'ASH'])[cible].median()
    pred = df.iloc[idx_te].set_index(['STATUT_JUR', 'ASH']).index.map(med)
    scores.append({'Modèle': 'Baseline (médiane statut x ASH)', 'pli': pli,
                   'MAE': mean_absolute_error(y_te, pred), 'R2': r2_score(y_te, pred)})

    # Médiane globale
    pred = np.full(len(y_te), y_tr.median())
    scores.append({'Modèle': 'Médiane globale', 'pli': pli,
                   'MAE': mean_absolute_error(y_te, pred), 'R2': r2_score(y_te, pred)})

    # Les modèles
    for nom, (creer, cols, en_log) in modeles_cv.items():
        modele = creer()
        if en_log:
            modele.fit(X_tr[cols], np.log(y_tr))
            pred = np.exp(modele.predict(X_te[cols]))
        else:
            modele.fit(X_tr[cols], y_tr)
            pred = modele.predict(X_te[cols])
        scores.append({'Modèle': nom, 'pli': pli,
                       'MAE': mean_absolute_error(y_te, pred), 'R2': r2_score(y_te, pred)})

# Moyenne et écart-type sur les 5 plis, du meilleur au moins bon
cv = (pd.DataFrame(scores).groupby('Modèle')
      .agg(MAE_moyenne=('MAE', 'mean'), MAE_ecart_type=('MAE', 'std'), R2_moyen=('R2', 'mean'))
      .sort_values('MAE_moyenne'))
cv.round({'MAE_moyenne': 0, 'MAE_ecart_type': 0, 'R2_moyen': 2})

,MAE_moyenne,MAE_ecart_type,R2_moyen
Modèle,,,
"Gradient Boosting, cible en log",161.0,13.0,0.79
Gradient Boosting,162.0,13.0,0.79
Random Forest,164.0,14.0,0.79
"Régression linéaire, toutes variables",180.0,14.0,0.74
Baseline (médiane statut x ASH),229.0,23.0,0.52
"Régression linéaire, loyer seul",296.0,27.0,0.39
Médiane globale,352.0,48.0,-0.11


In [11]:
# Le log gagne-t-il à chaque pli ? Écart de MAE pli par pli (positif = le log fait mieux)
par_pli = pd.DataFrame(scores).pivot(index='pli', columns='Modèle', values='MAE')
(par_pli['Gradient Boosting'] - par_pli['Gradient Boosting, cible en log']).round(1)

pli
0    3.2
1   -0.1
2    0.4
3    1.4
4    1.9
dtype: float64

> **Résultat :**
> - Sur 5 découpages, le classement est stable : médiane globale (352 €), baseline (229 €), régression linéaire (180 €), puis les modèles à arbres autour de 160 €.
> - Le Gradient Boosting avec la cible en log obtient **161 € ± 13 €** (R² moyen de 0,79). Le log fait mieux que le Gradient Boosting classique sur 4 plis sur 5, mais le gain moyen n'est que de 1 €.
> - L'écart-type entre plis (environ 13 €) est du même ordre que les écarts entre modèles à arbres : il faut rester prudent sur leur classement. **On retient le Gradient Boosting en log**, car il est au moins aussi bon que les autres, surapprend moins que la forêt, et sa cible en log est cohérente avec l'asymétrie du prix.

## 8. Analyse des EHPAD les plus mal prédits (meme démarche, utilisation de l'IA pour identifier les faiblesses de mon modèle)

On compare, pour chaque EHPAD du test, le prix réel aux prédictions de trois modèles : la régression linéaire, la Random Forest et le modèle retenu (Gradient Boosting en log).

In [12]:
# 1. Régression linéaire sur toutes les variables (= le modèle 4)
lr = LinearRegression().fit(train[variables], y_train)

# 2. Tableau des résultats : identifiant, prix réel et prédiction de chaque modèle
resultats = test[['finesset', cible]].copy()
resultats['pred_lr'] = lr.predict(test[variables])
resultats['pred_rf'] = rf.predict(test[variables])
resultats['pred_gb'] = np.exp(gb_log.predict(test[variables]))   # modèle retenu, remis en euros

# 3. Erreur de chaque modèle : prix réel - prix prédit
# positive = le modèle a sous-estimé ; négative = il a surestimé
for m in ['lr', 'rf', 'gb']:
    resultats['err_' + m] = resultats[cible] - resultats['pred_' + m]

# 4. Récupérer statut, ASH et département en clair (ils sont encodés dans test)
resultats = resultats.merge(df[['finesset', 'STATUT_JUR', 'ASH', 'DEP']], on='finesset')

# 5. Les 10 EHPAD les plus mal prédits par chaque modèle
for m in ['lr', 'rf', 'gb']:
    print('Modèle', m)
    ordre = resultats['err_' + m].abs().sort_values(ascending=False).index   # du plus grand écart au plus petit
    display(resultats.loc[ordre, ['finesset', cible, 'pred_' + m, 'err_' + m,
                                  'STATUT_JUR', 'ASH', 'DEP']].head(10).round(0))

# 6. Les 20 pires EHPAD sont-ils les mêmes pour les trois modèles ?
pires = {m: set(resultats['err_' + m].abs().nlargest(20).index) for m in ['lr', 'rf', 'gb']}
print('EHPAD communs aux 20 pires des 3 modèles :', len(pires['lr'] & pires['rf'] & pires['gb']))

# 7. Erreur moyenne de chaque modèle selon le statut juridique
erreurs = resultats[['err_lr', 'err_rf', 'err_gb']].abs()
erreurs['STATUT_JUR'] = resultats['STATUT_JUR']
display(erreurs.groupby('STATUT_JUR').mean().round(0))

# 8. Nombre d'EHPAD avec plus de 500 € d'erreur, par modèle
print((resultats[['err_lr', 'err_rf', 'err_gb']].abs() > 500).sum())

Modèle lr


,finesset,PRIX_MOIS_HEBPERM_CS_GIR56,pred_lr,err_lr,STATUT_JUR,ASH,DEP
1203,920803103,5383.0,3590.0,1793.0,3-Privé commercial,NON,92
1137,860786102,3396.0,1897.0,1499.0,2-Privé non lucratif,OUI,86
1013,750831224,2536.0,3823.0,-1287.0,2-Privé non lucratif,NON,75
808,590815882,3400.0,2192.0,1208.0,2-Privé non lucratif,NON,59
765,590007266,3394.0,2268.0,1127.0,2-Privé non lucratif,NON,59
197,160004370,1410.0,2486.0,-1076.0,3-Privé commercial,NON,16
1007,750041436,5193.0,4122.0,1070.0,3-Privé commercial,NON,75
218,170804041,2944.0,1875.0,1069.0,2-Privé non lucratif,OUI,17
1011,750800559,4375.0,3377.0,998.0,2-Privé non lucratif,OUI,75
220,170805642,1580.0,2517.0,-937.0,3-Privé commercial,NON,17


Modèle rf


,finesset,PRIX_MOIS_HEBPERM_CS_GIR56,pred_rf,err_rf,STATUT_JUR,ASH,DEP
1203,920803103,5383.0,3541.0,1842.0,3-Privé commercial,NON,92
1011,750800559,4375.0,2758.0,1617.0,2-Privé non lucratif,OUI,75
1137,860786102,3396.0,1891.0,1505.0,2-Privé non lucratif,OUI,86
1177,910460096,2462.0,3859.0,-1397.0,3-Privé commercial,NON,91
808,590815882,3400.0,2332.0,1069.0,2-Privé non lucratif,NON,59
765,590007266,3394.0,2421.0,974.0,2-Privé non lucratif,NON,59
1007,750041436,5193.0,4252.0,941.0,3-Privé commercial,NON,75
218,170804041,2944.0,2016.0,927.0,2-Privé non lucratif,OUI,17
1029,770802668,2009.0,2917.0,-908.0,3-Privé commercial,OUI,77
940,690041074,3182.0,2274.0,908.0,2-Privé non lucratif,OUI,69


Modèle gb


,finesset,PRIX_MOIS_HEBPERM_CS_GIR56,pred_gb,err_gb,STATUT_JUR,ASH,DEP
1203,920803103,5383.0,3462.0,1921.0,3-Privé commercial,NON,92
1011,750800559,4375.0,2639.0,1736.0,2-Privé non lucratif,OUI,75
1137,860786102,3396.0,2019.0,1377.0,2-Privé non lucratif,OUI,86
1177,910460096,2462.0,3823.0,-1362.0,3-Privé commercial,NON,91
808,590815882,3400.0,2397.0,1003.0,2-Privé non lucratif,NON,59
1005,750003972,3609.0,4570.0,-961.0,3-Privé commercial,NON,75
218,170804041,2944.0,2064.0,880.0,2-Privé non lucratif,OUI,17
1029,770802668,2009.0,2860.0,-851.0,3-Privé commercial,OUI,77
64,060798915,2775.0,3619.0,-844.0,3-Privé commercial,NON,06
940,690041074,3182.0,2367.0,815.0,2-Privé non lucratif,OUI,69


EHPAD communs aux 20 pires des 3 modèles : 11


,err_lr,err_rf,err_gb
STATUT_JUR,,,
1-Public,121.0,104.0,103.0
2-Privé non lucratif,171.0,169.0,154.0
3-Privé commercial,280.0,259.0,259.0


err_lr    63
err_rf    58
err_gb    49
dtype: int64


> **Analyse :**
> - **Les trois modèles se trompent sur les mêmes établissements :** 11 EHPAD sont communs à leurs 20 pires prédictions. Ces établissements ont quelque chose que les données ne décrivent pas.
> - **Les plus grosses erreurs concernent surtout le privé commercial et l'Île-de-France**, avec des écarts dépassant parfois 1 900 €.
> - **L'erreur dépend fortement du statut :** pour le modèle retenu, environ 105 € pour le public, 155 € pour l'associatif, 260 € pour le commercial. Les tarifs publics sont encadrés ; le prix des commerciaux dépend du standing et des services, absents des données.
> - **Les erreurs vont dans les deux sens :** le modèle sous-estime des établissements haut de gamme et surestime des commerciaux plus modestes situés en zone chère.

## 9. Conclusion

Erreurs moyennes sur **5 découpages par gestionnaire** (validation croisée, section 7) :

| Modèle | MAE (moyenne ± écart-type) | R² moyen |
|---|---|---|
| Prédire le prix médian pour tous | 352 € ± 48 € | — |
| Baseline (médiane statut × ASH) | 229 € ± 23 € | 0,52 |
| Régression linéaire, loyer seul | 296 € ± 27 € | 0,39 |
| Régression linéaire, toutes variables | 180 € ± 14 € | 0,74 |
| Random Forest | 164 € ± 14 € | 0,79 |
| Gradient Boosting | 162 € ± 13 € | 0,79 |
| **Gradient Boosting, cible en log (modèle final)** | **161 € ± 13 €** | **0,79** |

**Modèle retenu :** Gradient Boosting entraîné sur le log du prix. Il prédit le prix mensuel **à environ 161 € près en moyenne, soit environ 7,4 % du prix moyen**, sur des établissements de gestionnaires qu'il n'a jamais vus. Il divise par plus de deux l'erreur d'une prédiction naïve et réduit de 30 % celle de la baseline.

**Ce qui fait le prix :** d'abord le statut juridique, puis l'immobilier local (loyer) et le département. Les autres variables n'apportent qu'un complément.

**Usage possible :** au-delà de la prédiction, le modèle permet de repérer les établissements dont le prix s'écarte nettement de ce que leur profil laisse attendre.

## 10. Limites et pistes

**Limites**
- Les données ne décrivent ni le standing, ni les services, ni l'état du bâtiment : c'est ce qui explique l'essentiel des grosses erreurs, surtout dans le privé commercial.
- Les données datent de 2019 : les prix ont évolué depuis.
- Repasser du log aux euros avec l'exponentielle donne plutôt une prédiction « médiane » : le modèle a tendance à sous-estimer légèrement les prix moyens.

**Pistes**
- Donner le département au Gradient Boosting comme variable catégorielle (`categorical_features`) plutôt qu'en 95 colonnes one-hot.
- Calculer des fourchettes de prix à partir des erreurs observées (par exemple par statut).
- Prédire le prix d'EHPAD fictifs pour illustrer l'effet de chaque variable.